In [ ]:
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import re
import datetime
import statsmodels.api as sm

# Loading Data

In [ ]:
All_news_proccesed_merged = pd.read_excel(r'../../Data/All_news_FBRT_adj_processed.xlsx')

In [ ]:
# All_news_proccesed_merged = pd.read_excel(r'../../Data/All_news_my_LLM_mapped_processed.xlsx')

In [ ]:
# funds_data=pd.read_excel(r'../../Data/funds_data.xlsx')

In [ ]:
funds_data = pd.read_csv(r'../../Data/Y/Y_interpolated.csv', encoding='utf-16', sep = '\t')
funds_data['date'] = pd.to_datetime(funds_data['date'])

In [ ]:
# funds_data = pd.read_csv(r'../../Data/Y/Y_shrinked_quart.csv', encoding='utf-16', sep = '\t')
# funds_data['date'] = pd.to_datetime(funds_data['date'])

# Fund assets dynamics 

## Correspondence of names

In [ ]:
names_in_funds = funds_data[["Конечный актив", "ISIN конечного актива"]]
# names_in_funds = names_in_funds[names_in_funds["Конечный актив"].str.contains("акция", case=False, na=False)]

names_in_funds = names_in_funds[names_in_funds["Конечный актив"].str.count(',') >= 1]

names_in_funds["Asset_type"] = names_in_funds["Конечный актив"].astype(str).apply(lambda x: x.split(',')[1])
names_in_funds["Company"] = names_in_funds["Конечный актив"].astype(str).apply(lambda x: x.split(',')[0])
names_in_funds = names_in_funds.drop_duplicates(subset="ISIN конечного актива")

In [ ]:
df_news_names = pd.read_csv('names.csv', encoding='utf-16', sep= '^').rename(columns={"0": "source_file"})

company_type_isin = pd.merge(
    df_news_names.assign(key=1, tmp=df_news_names['source_file'].str.lower()),
    names_in_funds.assign(key=1, tmp=names_in_funds['Company'].str.lower()),
    on="key", suffixes=('_left', '_right')
).drop("key", axis=1)

company_type_isin = company_type_isin[
    company_type_isin.apply(
        lambda row: row['tmp_left'] in row['tmp_right'] or row['tmp_right'] in row['tmp_left'],
        axis=1
    )
].drop(columns=["tmp_left", "tmp_right",'source_file'])
company_type_isin.to_excel('company_type_isin1.xlsx', index=False)

In [ ]:
company_type_isin = pd.read_excel('company_type_isin.xlsx')

## News labling and classification

In [ ]:
import re
import difflib
from joblib import Parallel, delayed
from tqdm import tqdm

# 1) Your expanded keyword lists
forecast_keywords = [
    "прогноз", "ожидается", "предсказывают", "аналитики ожидают",
    "по прогнозам", "прогнозируется", "перспектив", "прогнозный",
    "прогнозирование", "планируется", "рассчитывают", "предполагается",
    "вероятностное", "ожидания", "тенденция", "прогноз роста",
    "прогноз прибыли", "ориентир", "расчёт", "меры стимулирования",
    "оптимистичный", "пессимистичный"
]

earnings_keywords = [
    "отчётность", "финансовый отчёт", "баланс", "консолидированный отчёт",
    "отчёт о прибылях", "отчёт о убытках", "отчёт о движении денежных средств",
    "годовой отчёт", "квартальный отчёт", "финансовые результаты",
    "выручка", "чистая прибыль", "рентабельность", "ebitda", "eps",
    "дивиденды", "активы", "обязательства", "кассовые потоки",
    "мсфо", "ifrs", "показатели", "анализ отчётности"
]

sanctions_keywords = [
    "санкци", "ограничени", "рестрикци", "эмбарго", "заморозка активов",
    "блокировк счетов", "запрет", "штраф", "экономические меры",
    "санкционный", "секторальные санкции", "торговые ограничения",
    "визовые ограничения", "торговое эмбарго", "финансовые санкции",
    "торговый запрет", "замораживают активы", "исключение из swift",
    "ограничение импорта", "ограничение экспорта"
]

# 2) Compile exact‐match regexes
def make_pattern(keywords):
    esc = [re.escape(w) for w in keywords]
    return re.compile(r'\b(' + '|'.join(esc) + r')\b', flags=re.IGNORECASE)

forecast_pat  = make_pattern(forecast_keywords)
earnings_pat  = make_pattern(earnings_keywords)
sanctions_pat = make_pattern(sanctions_keywords)

# 3) Fuzzy‐match helper
def fuzzy_count(text, keywords, threshold=0.8):
    tokens = re.findall(r'\b\w+\b', text.lower())
    cnt = 0
    for kw in keywords:
        if re.search(rf'\b{re.escape(kw)}\b', text, re.IGNORECASE):
            continue
        for tok in tokens:
            if difflib.SequenceMatcher(None, kw.lower(), tok).ratio() >= threshold:
                cnt += 1
                break
    return cnt

# 4) Single‐text classifier
def classify(text):
    f = len(forecast_pat.findall(text)) + fuzzy_count(text, forecast_keywords)
    e = len(earnings_pat.findall(text)) + fuzzy_count(text, earnings_keywords)
    s = len(sanctions_pat.findall(text)) + fuzzy_count(text, sanctions_keywords)
    best = max(('forecast', f), ('earnings', e), ('sanctions', s), key=lambda x: x[1])
    return best[0] if best[1] > 0 else 'other'

# 5) Parallel application with joblib + tqdm
texts = All_news_proccesed_merged['news_content'].fillna('').tolist()
news_types = Parallel(n_jobs=-1)(
    delayed(classify)(txt) for txt in tqdm(texts, desc="Classifying news", total=len(texts))
)

All_news_proccesed_merged['news_type'] = news_types


for finbert

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd

names_list = company_type_isin['Company'].tolist()
l = 0.8

def process_row(row):
    results = []
    news_content = row.get('news_content', '')
    if not news_content or pd.isnull(news_content):
        return results
    for match in names_list:
        if match[0:int(len(match)*l)] in news_content:
            results.append({**row, 'firm_in_text': match, 'n_matches': news_content.count(match)})
    return results

# Convert rows to dictionaries for safe pickling in joblib subprocesses
rows = [row.to_dict() for _, row in All_news_proccesed_merged.iterrows()]

# Process rows in parallel using all available CPUs
results = Parallel(n_jobs=-1)(
    delayed(process_row)(row) for row in tqdm(rows, total=len(rows), desc="Processing rows")
)

# Flatten the list of lists and create a new DataFrame
flat_results = [item for sublist in results for item in sublist]
All_news_proccesed_merged = pd.DataFrame(flat_results).drop_duplicates()

for my llm

In [ ]:
# l = 0.8
# All_news_proccesed_merged['n_matches'] = All_news_proccesed_merged.apply(
#     lambda row: row['news_content'].lower().count(
#         row['firm_in_text'][:int(len(row['firm_in_text']) * l)].lower()
#     ) if pd.notnull(row['news_content']) and pd.notnull(row['firm_in_text']) else 0,
#     axis=1
# )

for BoW

In [ ]:
# from joblib import Parallel, delayed
# from tqdm import tqdm
# import pandas as pd

# names_list = company_type_isin['Company'].tolist()
# l = 0.8

# def process_row(row):
#     results = []
#     news_content = row.get('news_content', '')
#     if not news_content or pd.isnull(news_content):
#         return results
#     for match in names_list:
#         if match[0:int(len(match)*l)] in news_content:
#             results.append({**row, 'firm_in_text': match, 'n_matches': news_content.count(match)})
#     return results

# # Convert rows to dictionaries for safe pickling in joblib subprocesses
# rows = [row.to_dict() for _, row in All_news_proccesed_merged.iterrows()]

# # Process rows in parallel using all available CPUs
# results = Parallel(n_jobs=-1)(
#     delayed(process_row)(row) for row in tqdm(rows, total=len(rows), desc="Processing rows")
# )

# # Flatten the list of lists and create a new DataFrame
# flat_results = [item for sublist in results for item in sublist]
# All_news_proccesed_merged = pd.DataFrame(flat_results).drop_duplicates()

In [ ]:
news_types_lst = ['отчетность', 'прогноз', 'рынок', 'санкции', 'мосбиржа', 'акции']
l = 0.75

for word in news_types_lst:
    All_news_proccesed_merged[word] = All_news_proccesed_merged['news_content'].apply(
        lambda s: s.lower().count(word.lower()[0:int(len(word)*l)]) if isinstance(s, str) else 0
    )
# убираю неполные совпадения
# All_news_proccesed_merged = All_news_proccesed_merged[All_news_proccesed_merged['n_matches'] != 0]

In [ ]:
# All_news_proccesed_merged = All_news_proccesed_merged.iloc[:, :-8]

In [ ]:
dummies = pd.get_dummies(
    All_news_proccesed_merged['news_type'],
    prefix='',
    prefix_sep='',
    dtype=int
)
All_news_proccesed_merged = pd.concat(
    [All_news_proccesed_merged, dummies],
    axis=1
)


In [ ]:
# All_news_proccesed_merged.to_excel(
#     r'../../Data/All_news_FBRT_adj_classes.xlsx',
#     index=False,
#     engine='xlsxwriter',
#     engine_kwargs={'options': {'strings_to_urls': False}}
# )

## News pockets sorting

In [ ]:
All_news_proccesed_merged = pd.read_excel(r'../../Data/All_news_FBRT_adj_processed.xlsx')

In [ ]:
all_date_cols = funds_data['date'].dt.strftime('%Y-%m-%d').drop_duplicates().tolist()

news_df = All_news_proccesed_merged.copy()
news_df['date'] = pd.to_datetime(news_df['date'], format='%Y-%m-%d')

bins = sorted(all_date_cols, key=lambda d: datetime.datetime.strptime(d, "%Y-%m-%d"))
edges = [datetime.datetime.strptime(d, "%Y-%m-%d") for d in bins]
labels = [f"{edges[i+1].strftime('%Y-%m-%d')}:{edges[i].strftime('%Y-%m-%d')}" for i in range(len(edges)-1)]

news_df.insert(column='interval', value= pd.cut(news_df['date'], bins=edges, labels=labels, include_lowest=True), loc=4)

In [ ]:
# count_pivot = news_df.pivot_table(
#     index=['interval'],
#     columns='sentiment',
#     aggfunc='size',
#     fill_value=0
# )


In [ ]:
# # Сводная таблица для суммирования значений по столбцам n_matches и отчетность
# sum_pivot = news_df.pivot_table(
#     index=['interval'],
#     values=['отчетность', 'прогноз', 'рынок', 'санкции', 'мосбиржа', 'акции'],
#     aggfunc='sum',
#     fill_value=0
# )

In [ ]:
merged_pivot = count_pivot.reset_index().merge(sum_pivot.reset_index(), on=['interval'])
merged_pivot.insert(column='start_date', value=pd.to_datetime(merged_pivot['interval'].apply(lambda x: x.split(':')[1])).dt.date, loc=1)
merged_pivot.insert(column='end_date', value=pd.to_datetime(merged_pivot['interval'].apply(lambda x: x.split(':')[0])).dt.date, loc=2)
merged_pivot['news_count'] = merged_pivot['Positive'] + merged_pivot['Negative'] + merged_pivot['Neutral']
merged_pivot['net_sent_score'] = (merged_pivot['Positive'] - merged_pivot['Negative'])/(merged_pivot['Positive'] + merged_pivot['Negative'] + merged_pivot['Positive'])

In [ ]:
# merged_pivot.to_excel('EDA_FBRT_adj.xlsx', index=False, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}})

In [ ]:
# Сводная таблица для подсчёта количества новостей по sentiment
count_pivot = news_df.pivot_table(
    index=['firm_in_text','interval'],
    columns='sentiment',
    aggfunc='size',
    fill_value=0
)

# Сводная таблица для суммирования значений по столбцам n_matches и отчетность
sum_pivot = news_df.pivot_table(
    index=['firm_in_text','interval'],
    values=['n_matches', 'отчетность', 'прогноз', 'рынок', 'санкции', 'мосбиржа', 'акции'],
    aggfunc='sum',
    fill_value=0
)

# Объединяем обе сводные таблицы по индексу (обнуляем индекс для объединения)
merged_pivot = count_pivot.reset_index().merge(sum_pivot.reset_index(), on=['firm_in_text','interval'])
merged_pivot.insert(column='start_date', value=pd.to_datetime(merged_pivot['interval'].apply(lambda x: x.split(':')[1])).dt.date, loc=2)
merged_pivot.insert(column='end_date', value=pd.to_datetime(merged_pivot['interval'].apply(lambda x: x.split(':')[0])).dt.date, loc=3)

In [ ]:
# news_panel_X = pd.merge(company_type_isin, merged_pivot, left_on='Company',right_on='firm_in_text', how="right").drop_duplicates()
# news_panel_X.to_excel(r'../../Data/X/News_panel_FBRT_adj_ru_.xlsx', index=False)

In [ ]:
import pandas as pd
import datetime

# your existing code
all_date_cols = funds_data['date'].dt.strftime('%Y-%m-%d')\
    .drop_duplicates().tolist()

news_df = All_news_proccesed_merged.copy()
news_df['date'] = pd.to_datetime(news_df['date'], format='%Y-%m-%d')

bins = sorted(all_date_cols,
              key=lambda d: datetime.datetime.strptime(d, "%Y-%m-%d"))
edges = [datetime.datetime.strptime(d, "%Y-%m-%d") for d in bins]
labels = [
    f"{edges[i+1].strftime('%Y-%m-%d')}:{edges[i].strftime('%Y-%m-%d')}" 
    for i in range(len(edges)-1)
]

news_df.insert(
    column='interval',
    value=pd.cut(news_df['date'], bins=edges,
                 labels=labels, include_lowest=True),
    loc=4
)

# now build the pivot counts:
pivot = (
    news_df
    .groupby(['firm_in_text', 'interval', 'news_type', 'sentiment'])
    .size()
    .unstack(['news_type','sentiment'], fill_value=0)
)

# flatten the MultiIndex columns:
pivot.columns = [
    f"{ntype}_{sent}" 
    for ntype, sent in pivot.columns
]
pivot.reset_index(inplace=True)
pivot.insert(column='start_date', value=pd.to_datetime(pivot['interval'].apply(lambda x: x.split(':')[1])).dt.date, loc=2)
pivot.insert(column='end_date', value=pd.to_datetime(pivot['interval'].apply(lambda x: x.split(':')[0])).dt.date, loc=3)
# pivot is now a DataFrame indexed by (source_file, interval)
# with columns like: forecast_Positive, forecast_Neutral, ..., other_Negative


In [ ]:
# news_panel_X = pd.merge(company_type_isin, pivot, left_on='Company',right_on='firm_in_text', how="right").drop_duplicates()
# news_panel_X.to_excel(r'../../Data/X/News_panel_FBRT_adj_ru_class_shrinked_quart.xlsx', index=False)

## Funds changes of portfolios

In [ ]:
unique_columns = []
for df in dict_df_wide.values():
    for col in df.columns:
        if col not in unique_columns:
            unique_columns.append(col)
non_date_columns = unique_columns[:7]
date_columns = unique_columns[7:]
date_columns_sorted = sorted(date_columns, key=lambda d: datetime.datetime.strptime(d, "%Y-%m-%d"), reverse=True)
unique_columns_sorted = non_date_columns + date_columns_sorted
print(unique_columns_sorted)


In [ ]:
all_date_cols = set()
for df in dict_df_wide.values():
    all_date_cols.update([col for col in df.columns[7:] if re.match(r"\d{4}-\d{2}-\d{2}", col)])
all_date_cols = sorted(all_date_cols, key=lambda d: datetime.datetime.strptime(d, "%Y-%m-%d"), reverse=True)

funds_dict = {}
for key, df in dict_df_wide.items():
    general = df.iloc[:, :7]
    current_dates = [col for col in df.columns[7:] if re.match(r"\d{4}-\d{2}-\d{2}", col)]
    date_part = df[current_dates].reindex(columns=all_date_cols)
    funds_dict[key] = pd.concat([general, date_part.bfill(axis=1)], axis=1)

In [ ]:
final_diff_dict = {}
for key, df in funds_dict.items():
    general = df.iloc[:, :7]
    date_part = df.iloc[:, 7:]
    
    diff_df = {}
    for i in range(len(date_part.columns) - 1):
        col_name = f"{date_part.columns[i]}:{date_part.columns[i+1]}"
        diff_df[col_name] = date_part.iloc[:, i] - date_part.iloc[:, i+1]
    diff_df = pd.DataFrame(diff_df, index=df.index)
    
    final_diff_dict[key] = pd.concat([general, diff_df], axis=1)

In [ ]:
final_diff_dict['Альфа-Капитал Управляемые акции__4213']

## X and Y

In [ ]:
final_diff_df = pd.concat(final_diff_dict.values(), ignore_index=True)
final_diff_df.drop(columns=['Название фонда','Номер регистрации', 'ISIN Фонда', 'Название УК', 'Класс активов','Конечный актив'],inplace=True)
final_diff_df_pooled = final_diff_df.groupby(by = 'ISIN конечного актива').sum().reset_index()

final_diff_df_pooled = final_diff_df_pooled[
    final_diff_df_pooled['ISIN конечного актива'].isin(news_panel_X['ISIN конечного актива'])
]
 

In [ ]:
data_Y = final_diff_df_pooled.melt(
    id_vars=final_diff_df_pooled.columns[:1],
    var_name='interval',
    value_name='Y'  # можно заменить 'value' на другое название для обозначения изменений
)
data_X = news_panel_X[['Конечный актив', 'ISIN конечного актива', 'interval', 'Negative', 'Neutral', 'Positive']]

In [ ]:
data_X.to_csv('pooled_data_X.csv', encoding='utf-16', index=False)
data_Y.to_csv('pooled_data_Y.csv', encoding='utf-16', index=False)

In [ ]:
data_X_Y = pd.merge(data_X, data_Y, on=['ISIN конечного актива','interval'], how='inner').sort_values(by=['Конечный актив','interval'], ascending=[True, False])

In [ ]:
data_X_Y.to_csv('pooled_data_X_Y.csv', encoding='utf-16', index=False)

# Pooled model

In [ ]:
pooled_data_X_Y = data_X_Y.groupby(by = 'interval').sum().reset_index()

In [ ]:
X = pooled_data_X_Y[['Negative', 'Neutral', 'Positive']]
y = pooled_data_X_Y['Y']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

# Each instrument separately

In [ ]:
results_dict = {}
for isin, group in data_X_Y.groupby("ISIN конечного актива"):
    X = group[['Negative', 'Neutral', 'Positive']]
    y = group['Y']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    results_dict[isin] = model


In [ ]:
# Находим ISIN с максимальным R^2
max_isin, max_model = max(results_dict.items(), key=lambda item: item[1].rsquared)
print(f"Model with highest R-squared: ISIN {max_isin}")
print("R-squared:", max_model.rsquared)
print(max_model.summary())


In [ ]:
significant_models = {}
for isin, model in results_dict.items():
    pvals = model.pvalues.drop('const', errors='ignore')
    if any(p < 0.05 for p in pvals):
        significant_models[isin] = model

print("Models with at least one statistically significant coefficient:")
for isin, model in significant_models.items():
    print(f"ISIN: {isin}, R-squared: {model.rsquared:.4f}")
    print(model.summary())

# FE model

In [ ]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

FE_data_X_Y = data_X_Y
FE_data_X_Y['time'] = pd.to_datetime(FE_data_X_Y['interval'].apply(lambda s: s.split(':')[0]))
FE_data_X_Y = FE_data_X_Y.set_index(['ISIN конечного актива', 'time'])
exog = sm.add_constant(FE_data_X_Y[['Negative', 'Neutral', 'Positive']])

model = PanelOLS(FE_data_X_Y['Y'], exog, entity_effects=True)

results = model.fit()
print(results.summary)

# Other

## Stocks quotes

In [ ]:
all_instr_quotes_dict['IMOEX']['TRADEDATE']

In [ ]:
instruments = pd.read_csv('Export_ru_securities-list_20250203.csv', encoding='cp1251')
instruments = instruments[instruments['TRADE_CODE'].isin(instruments['TRADE_CODE'].dropna())]

instruments = instruments[instruments['SUPERTYPE']=='Акции']

In [ ]:
n = 17

id = instruments['TRADE_CODE'].iloc[n]
df = all_instr_quotes_dict[id]
df = df[df['BOARDID']=='TQBR']

plt.figure(figsize=(12, 6))
plt.plot(pd.to_datetime(df["TRADEDATE"]), df["LEGALCLOSEPRICE"], color = 'blue')
# plt.plot(pd.to_datetime(all_instr_quotes_dict['IMOEX']['TRADEDATE']), all_instr_quotes_dict['IMOEX']['CLOSE'], color = 'red')
plt.xlabel("Trade Date")
plt.ylabel("Legal Close Price")
plt.title(f"{instruments['SUPERTYPE'].iloc[n]+' '+instruments['EMITENT_FULL_NAME'].iloc[n]}")
plt.xticks(rotation=45)
plt.grid()
plt.show()


## Stocks names in funds assets

In [ ]:
stocks = df[df['Класс активов']== 'russian_stocks']['Конечный актив'].unique()
stocks = pd.Series(stocks).dropna()
stocks = stocks.str.split(',').str[0]
stocks = stocks[stocks.str.contains(r'^[а-яА-ЯёЁ]+$', na=False)]

In [ ]:
stocks.to_csv('rus_stocks_names.txt', index=False, header=False)

# Plots

In [ ]:
merged_pivot_c = merged_pivot.groupby('end_date').sum(numeric_only=True).reset_index()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Aggregate the data by 'end_date' summing the numeric columns (Positive, Neutral, Negative)
merged_pivot_agg = merged_pivot.groupby('end_date', as_index=False).sum(numeric_only=True)

# Ensure the end_date column is in datetime format and sort by date
merged_pivot_agg['end_date'] = pd.to_datetime(merged_pivot_agg['end_date'])
merged_pivot_agg = merged_pivot_agg.sort_values('end_date')

# Define the sentiment areas and their colors
areas = ['Positive', 'Negative', 'Neutral']
colors = {
    "Positive": "green",
    "Neutral": "gray",
    "Negative": "red"
}

# Create the stack plot using the aggregated data
fig, ax = plt.subplots(figsize=(12, 10))
ax.stackplot(
    merged_pivot_agg['end_date'],
    [merged_pivot_agg[area] for area in areas],
    labels=areas,
    colors=[colors[area] for area in areas],
    alpha=0.75
)

# Update font settings
plt.rcParams.update({'font.family': 'Times New Roman', 'font.size': 14})

# Set title and adjust the y-axis to display on the right side
ax.set_title('')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.legend(loc='upper left')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

merged_pivot_c = merged_pivot.copy()
merged_pivot_c['end_date'] = pd.to_datetime(merged_pivot_c['end_date'])
# mask = merged_pivot_c['end_date'] < pd.Timestamp('2018-04-10')
# merged_pivot_c.loc[mask, ["Positive", "Neutral", "Negative"]]
merged_pivot_c = merged_pivot_c.sort_values('end_date')

areas = ['Positive', 'Neutral', 'Negative']
areas_ordered = sorted(areas, key=lambda c: merged_pivot_c[c].sum(), reverse=True)

# Simple color names for each sentiment
colors = {
    "Positive": "green",
    "Neutral": "gray",
    "Negative": "red"
}

fig, ax = plt.subplots(figsize=(12, 7))
for col in areas_ordered:
    ax.fill_between(merged_pivot_c['end_date'], merged_pivot_c[col],
                    color=colors[col], alpha=0.75, label=col)

# ax.set_xlabel('Date')
# ax.set_ylabel('Count')
plt.rcParams.update({'font.family': 'Times New Roman', 'font.size': 14})
ax.set_title('Number of news by sentiment category')
ax.yaxis.tick_right()          # Move y-axis ticks to the right
ax.yaxis.set_label_position("right")
ax.legend(loc='upper left')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

merged_pivot_c = merged_pivot.copy()
# Assume merged_pivot_c is already created/modified as in previous steps
merged_pivot_c['end_date'] = pd.to_datetime(merged_pivot_c['end_date'])
merged_pivot_c = merged_pivot_c.sort_values('end_date')

# Compute total count per row and convert each sentiment count into a percentage share
merged_pivot_c['total'] = merged_pivot_c[['Positive', 'Neutral', 'Negative']].sum(axis=1)
merged_pivot_c['Positive_pct'] = merged_pivot_c['Positive'] / merged_pivot_c['total'] * 100
merged_pivot_c['Neutral_pct']  = merged_pivot_c['Neutral']  / merged_pivot_c['total'] * 100
merged_pivot_c['Negative_pct'] = merged_pivot_c['Negative'] / merged_pivot_c['total'] * 100

# Plot a stacked area plot with the sentiment shares, where the total sums to 100%
fig, ax = plt.subplots(figsize=(12, 7))
ax.stackplot(merged_pivot_c['end_date'],
             merged_pivot_c['Positive_pct'],
             merged_pivot_c['Neutral_pct'],
             merged_pivot_c['Negative_pct'],
             labels=['Positive', 'Neutral', 'Negative'],
             colors=['green', 'gray', 'red'],
             alpha=0.75)

plt.rcParams.update({'font.family': 'Times New Roman', 'font.size': 14})
ax.yaxis.tick_right()          # Move y-axis ticks to the right

ax.set_title('Sentiment Share Percentages over Time (Total = 100%)')
ax.legend(loc='upper left')
ax.set_ylim(0, 100)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

merged_pivot_c = merged_pivot.copy()
merged_pivot_c['end_date'] = pd.to_datetime(merged_pivot_c['end_date'])
mask = merged_pivot_c['end_date'] < pd.Timestamp('2018-04-10')
merged_pivot_c.loc[mask, ["Positive", "Neutral", "Negative"]] /= 3
merged_pivot_c = merged_pivot_c.sort_values('end_date')

areas = ['Positive', 'Neutral', 'Negative']
areas_ordered = sorted(areas, key=lambda c: merged_pivot_c[c].sum(), reverse=True)

# Simple color names for each sentiment
colors = {
    "Positive": "green",
    "Neutral": "gray",
    "Negative": "red"
}

fig, ax = plt.subplots(figsize=(12, 7))
for col in areas_ordered:
    ax.plot(merged_pivot_c['end_date'], merged_pivot_c[col],
            color=colors[col], label=col, linewidth=2)

# ax.set_xlabel('Date')
# ax.set_ylabel('Count')
ax.set_title('Monthly number of news by sentiment category')
plt.rcParams.update({'font.family': 'Times New Roman', 'font.size': 14})
ax.yaxis.tick_right()          # Move y-axis ticks to the right

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.legend(loc='upper left')
plt.show()

# Models comparison

In [ ]:
Finbert = pd.read_excel(r'../../Data/All_news_FBRT_adj_processed.xlsx')

In [ ]:
Bow_ru = pd.read_excel(r'../../Data/All_news_BoW_ru_processed.xlsx')

In [ ]:
Bow = pd.read_excel(r'../../Data/All_news_BoW_processed.xlsx')

In [ ]:
df = pd.DataFrame({
    "finbert": Finbert["sentiment"],
    "bow":     Bow["sentiment"]
})

In [ ]:
# 1) Drop rows where either label is missing
df_clean = df.dropna(subset=["finbert", "bow"]).copy()

# 2) Optionally restrict strictly to known categories:
valid = ["Positive", "Neutral", "Negative"]
df_clean = df_clean[
    df_clean["finbert"].isin(valid) & 
    df_clean["bow"].isin(valid)
].copy()

# 3) Cast to string if necessary
df_clean["finbert"] = df_clean["finbert"].astype(str)
df_clean["bow"]     = df_clean["bow"].astype(str)

# 4) Compute raw confusion matrix (counts)
conf_mat = pd.crosstab(
    index=df_clean["finbert"],
    columns=df_clean["bow"],
    rownames=["FinBERT →"],
    colnames=["← BoW"]
)
print("Raw confusion matrix (counts):")
print(conf_mat, "\n")

# 5) Row‐normalized (fractions)
conf_mat_norm = conf_mat.div(conf_mat.sum(axis=1), axis=0)
print("Row‐normalized confusion matrix:")
print(conf_mat_norm, "\n")

In [ ]:
# 6) Overall accuracy & Cohen’s Kappa
accuracy = (df_clean["finbert"] == df_clean["bow"]).mean()
kappa    = cohen_kappa_score(df_clean["finbert"], df_clean["bow"])
print(f"Overall agreement (accuracy) = {accuracy:.3f}")
print(f"Cohen’s Kappa = {kappa:.3f}\n")

In [ ]:
import seaborn as sns
# 7) Plot row‐normalized confusion matrix as heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(
    conf_mat_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar=False,
    linewidths=0.5,
    linecolor="lightgray"
)
plt.title("Row‐Normalized Confusion Matrix\n(FinBERT vs. BoW)")
plt.ylabel("FinBERT label")
plt.xlabel("BoW label")
plt.tight_layout()
plt.show()

In [ ]:
conf_mat_colnorm = conf_mat.div(conf_mat.sum(axis=0), axis=1)
plt.figure(figsize=(6, 5))
sns.heatmap(
    conf_mat_colnorm,
    annot=True,
    fmt=".2f",
    cmap="Purples",
    cbar=False,
    linewidths=0.5,
    linecolor="lightgray"
)
plt.title("Column‐Normalized Confusion Matrix\n(BoW vs. FinBERT)")
plt.ylabel("FinBERT label")
plt.xlabel("BoW label")
plt.tight_layout()
plt.show()

In [ ]:
# 9) (Optional) Stacked‐bar: distribution of FinBERT‐labels within each BoW label
stacked = (
    df_clean
    .groupby("bow")["finbert"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)
stacked_pivot = (
    stacked
    .pivot(index="bow", columns="finbert", values="proportion")
    .fillna(0)
    .loc[valid, valid]
)
stacked_pivot.plot(
    kind="bar",
    stacked=True,
    color=["#D9534F","#999999","#5CB85C"],
    figsize=(6, 4)
)
plt.ylabel("Proportion of FinBERT Labels")
plt.xlabel("BoW Label")
plt.title("For each BoW label, FinBERT label distribution")
plt.legend(title="FinBERT →")
plt.tight_layout()
plt.show()